In [67]:
import pandas as pd
#loading data
main_demand = pd.read_excel('PGCB_date_power_demand.xlsx')
economs = pd.read_csv('economic_full_1.csv')
weather = pd.read_excel('weather_data.xlsx', header = 3)

In [68]:
#reading date, time
main_demand["datetime"] = pd.to_datetime(main_demand['datetime'])
weather['time'] = pd.to_datetime(weather['time'])
weather.rename(columns = {'time' : 'datetime'}, inplace = True)

In [69]:
#merging
data = pd.merge(main_demand, weather, on = 'datetime', how = 'left')

In [70]:
econom = economs.drop(columns = ["Country Name", "Indicator Code"], errors = 'ignore')
econom = econom.melt(
    id_vars = ['Indicator Name'],
    var_name = 'year',
    value_name = 'value')
economics = econom.pivot(
    index = 'year',
    columns = ['Indicator Name'],
    values = 'value').reset_index()
economics['year'] = economics['year'].astype(int)
data['year'] = data['datetime'].dt.year
data = pd.merge(data, economics, on = 'year', how = 'left')

In [71]:
data['datetime'] = pd.to_datetime(data['datetime'])
data['hour'] = data['datetime'].dt.hour
data['dayofweek'] = data['datetime'].dt.dayofweek
data['day'] = data['datetime'].dt.day
data['month'] = data['datetime'].dt.month
print(data.shape)

(92650, 1545)


In [72]:

data = data[["datetime", "load_shedding", 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'apparent_temperature (°C)', 'precipitation (mm)', 'wind_direction_10m (°)', 'cloud_cover (%)', 'sunshine_duration (s)', 'demand_mw', 'year']]
change = ["load_shedding", 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'apparent_temperature (°C)', 'precipitation (mm)', 'wind_direction_10m (°)', 'cloud_cover (%)', 'sunshine_duration (s)', 'demand_mw']
economics.columns = economics.columns.str.strip()
economics['GDP per capita (current US$)'] = economics['GDP per capita (current US$)'].shift(1)
economics['elec_pow_consu_kwh_per_capita'] = economics['Electric power consumption (kWh per capita)'].shift(1)
economics['population'] = (economics['Population, male'] + economics['Population, female']).shift(1)

In [73]:
#data cleaning, converting into hourly data 
data['datetime'] = pd.to_datetime(data['datetime'])
data = data.sort_values('datetime')
data = data.set_index('datetime')
data['time_diff'] = data.index.to_series().diff().shift(-1)
data['time_diff'] = data['time_diff'].dt.total_seconds()
data = data.dropna(subset = ['time_diff'])
data_hourly = pd.DataFrame()
for col in change:
    weighted_demand = data[col]*df['time_diff']
    weighted_sum = weighted_demand.resample('1h').sum()
    total_time = data['time_diff'].resample('1h').sum()
    data_hourly[col] = weighted_sum/total_time
data_hourly['demand_mw'] = data_hourly['demand_mw'].interpolate()
data_hourly = data_hourly.ffill()
data_hourly['year'] = data_hourly.index.year
data_hourly = pd.merge(data_hourly, economics[['GDP per capita (current US$)', 'elec_pow_consu_kwh_per_capita', 'population', 'year']], on = 'year', how = 'left')
data_hourly = data_hourly.ffill()


In [74]:
#setting lag, rolling features
data_hourly['lag_1'] = data_hourly['demand_mw'].shift(1)
data_hourly['lag_2'] = data_hourly['demand_mw'].shift(2)
data_hourly['lag_5'] = data_hourly['demand_mw'].shift(5)
data_hourly['lag_24'] = data_hourly['demand_mw'].shift(24)
data_hourly['lag_12'] = data_hourly['demand_mw'].shift(12)
data_hourly['lag_168'] = data_hourly['demand_mw'].shift(168)
data_hourly['lag_360'] = data_hourly['demand_mw'].shift(360)
data_hourly['lag_month'] = data_hourly['demand_mw'].shift(720)
data_hourly['lag_half_year'] = data_hourly['demand_mw'].shift(4320)

In [75]:
data_hourly["rolling_mean_4"] = data_hourly["demand_mw"].rolling(4).mean()
data_hourly["rolling_mean_24"] = data_hourly["demand_mw"].rolling(24).mean()
data_hourly["rolling_mean_168"] = data_hourly["demand_mw"].rolling(168).mean()
data_hourly["rolling_mean_360"] = data_hourly["demand_mw"].rolling(360).mean()
data_hourly["rolling_mean_month"] = data_hourly["demand_mw"].rolling(720).mean()

In [76]:
data_hourly["rolling_std_24"] = data_hourly["demand_mw"].rolling(24).std()
data_hourly["rolling_std_168"] = data_hourly["demand_mw"].rolling(168).std()
data_hourly["rolling_std_month"] = data_hourly["demand_mw"].rolling(720).std()

In [78]:
#spike handeling, splitting data to test and train
low = data_hourly['demand_mw'].quantile(0.02)
high = data_hourly['demand_mw'].quantile(0.98)
data_hourly['demand_mw'] = data_hourly["demand_mw"].clip(low, high)
data_hourly["target"] = data_hourly["demand_mw"].shift(-1)
#print(df.shape)
train_data = data_hourly[data_hourly['year'] <= 2023]
test_data = data_hourly[data_hourly['year'] == 2024]
features = ['temperature_2m (°C)', 'relative_humidity_2m (%)', 'apparent_temperature (°C)', 'wind_direction_10m (°)', 'sunshine_duration (s)', "lag_1",
                     'lag_2', 'lag_5', 'lag_24', 'lag_168', 'lag_360', 'rolling_mean_4', 'rolling_mean_24', "rolling_mean_168", "rolling_std_24", "rolling_std_168", "lag_12", 'population']
change1 =  ['temperature_2m (°C)', 'relative_humidity_2m (%)', 'apparent_temperature (°C)', 'wind_direction_10m (°)', 'sunshine_duration (s)', "lag_1",
                     'lag_2', 'lag_5', 'lag_24', 'lag_168', 'lag_360', 'rolling_mean_4', 'rolling_mean_24', "rolling_mean_168", "rolling_std_24", "rolling_std_168", "lag_12", "target", 'population']
train_data = train_data[change1].dropna()
test_data = test_data[change1].dropna()
train_X = train_data[features]
train_y = train_data['target']
test_X = test_data[features]
test_y = test_data["target"]
test_data.shape

(8784, 19)

In [79]:
#loading model, finding mape 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error
model = RandomForestRegressor(random_state = 42)
model.fit(train_X, train_y)
predict = model.predict(test_X)
error = mean_absolute_percentage_error(test_y, predict)
print(error)

0.033579502755750945


In [80]:
#feature significance
imp_features = pd.Series(model.feature_importances_, index = train_X.columns)
imp_features = imp.sort_values(ascending = False)
print(imp_features)

lag_1                        0.675578
rolling_mean_4               0.213714
lag_24                       0.036465
sunshine_duration (s)        0.013700
lag_12                       0.009796
temperature_2m (°C)          0.009057
lag_2                        0.007433
lag_5                        0.006712
lag_168                      0.005147
rolling_mean_24              0.004985
apparent_temperature (°C)    0.003236
relative_humidity_2m (%)     0.002847
rolling_mean_168             0.002749
rolling_std_24               0.002189
lag_360                      0.002095
rolling_std_168              0.001702
wind_direction_10m (°)       0.001482
population                   0.001113
dtype: float64
